## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-06-21-55-00 +0000,wsj,Basic Materials Roundup: Market Talk,https://www.wsj.com/business/basic-materials-r...
1,2026-11-06-21-55-00 +0000,wsj,Energy & Utilities Roundup: Market Talk,https://www.wsj.com/business/energy-utilities-...
2,2026-11-06-21-53-00 +0000,wsj,Financial Services Roundup: Market Talk,https://www.wsj.com/business/financial-service...
3,2026-11-06-21-53-00 +0000,wsj,"Tech, Media & Telecom Roundup: Market Talk",https://www.wsj.com/business/tech-media-teleco...
4,2026-11-06-21-50-35 +0000,chitri,Levi’s opens flagship store on Chicago’s Mag Mile,https://www.chicagotribune.com/2026/11/06/levi...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2514/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
140,trump,51
88,new,30
155,midterm,20
3,market,18
79,will,15


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
51,2026-11-06-20-55-00 +0000,wsj,Trump said that longtime adviser Stephen Mille...,https://www.wsj.com/politics/elections/trump-n...,168
27,2026-11-06-21-22-04 +0000,startribune,Stephen Miller replaces Susie Wiles as Trump's...,https://www.startribune.com/trump-replaces-sus...,165
116,2026-11-06-18-57-40 +0000,nypost,"Trump fires Susie Wiles after midterm blowout,...",https://nypost.com/2026/11/06/us-news/trump-fi...,164
98,2026-11-06-19-51-00 +0000,wsj,"Stephen Miller, who has been one of Trump’s mo...",https://www.wsj.com/politics/elections/in-tapp...,156
80,2026-11-06-20-11-45 +0000,bbc,Trump names Stephen Miller as new White House ...,https://www.bbc.co.uk/news/articles/cm9qzz5x9g...,154


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
51,168,2026-11-06-20-55-00 +0000,wsj,Trump said that longtime adviser Stephen Mille...,https://www.wsj.com/politics/elections/trump-n...
213,78,2026-11-06-15-22-00 +0000,wsj,The solid October jobs report leaves inflation...,https://www.wsj.com/economy/central-banking/so...
352,70,2026-11-06-03-11-54 +0000,nypost,Trump pins US stock market plunge on Democrats...,https://nypost.com/2026/11/05/us-news/trump-pi...
205,69,2026-11-06-15-32-25 +0000,chitri,Father of Meghan Markle arrested on charges re...,https://www.chicagotribune.com/2026/11/06/megh...
94,68,2026-11-06-19-56-00 +0000,wsj,Alphabet’s Drone Unit to Invest Up to $125 Mil...,https://www.wsj.com/business/alphabets-drone-u...
417,67,2026-11-05-22-48-00 +0000,wsj,The president signed a proclamation that he sa...,https://www.wsj.com/politics/policy/trump-sign...
420,60,2026-11-05-22-43-49 +0000,wapo,These Democratic Senate candidates won their r...,https://www.washingtonpost.com/politics/2026/1...
16,50,2026-11-06-21-32-54 +0000,latimes,Tesla will phase out Cybertruck production nex...,https://www.latimes.com/business/story/2026-11...
79,48,2026-11-06-20-13-24 +0000,wapo,"Poised for breakout moment in Latin America, P...",https://www.washingtonpost.com/world/2026/11/0...
98,48,2026-11-06-19-51-00 +0000,wsj,"Stephen Miller, who has been one of Trump’s mo...",https://www.wsj.com/politics/elections/in-tapp...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
